# 03.02 TQue 队列基础

## 小节概述

本节从一个 Tile 的所有权变化出发，解释 <code>TQue</code> 为什么同时承担缓冲管理与阶段同步，并重点区分 queue depth 与物理 Buffer 数量。

完成后你应能画出输入、输出 Tensor 的生命周期，计算三个队列的每核有效载荷，并说明为什么本实验固定 <code>depth=1</code>、使用 <code>num=1/2</code> 比较单缓冲和双缓冲。

<strong>建议用时：</strong> 30 分钟。<strong>本节产出：</strong> 默认参数的单/双缓冲资源推导与四道课后练习。


## 教程内容

### 1. 定位实验源码

下面的单元定位实验 3 目录并列出参考工程。<code>src/demo</code> 是完成版单/双缓冲对照工程，<code>src/practice</code> 是章节实践使用的学生工程。


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

previous_repo = globals().get('REPO_ROOT')
try:
    current = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    current = cached_repo.resolve()
search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([current, *current.parents])
REPO_ROOT = next(
    (p for p in search_roots if (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库内打开本 Notebook')
os.chdir(REPO_ROOT)
CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/03_double_buffer_pipeline'
DEMO = CHAPTER / 'src/demo'
ANSWER = CHAPTER / 'answer/03.02_queue_basics/answers.md'
ANSWER_CODE = CHAPTER / 'answer/03.02_queue_basics/queue_budget_practice.py'
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
PRACTICE_ROOT = USER_TEMP_ROOT / '03_double_buffer_pipeline'
PRACTICE_ROOT.mkdir(parents=True, exist_ok=True)
PRACTICE_FILE = PRACTICE_ROOT / 'queue_budget_practice.py'
print('practice file:', PRACTICE_FILE)
subprocess.run(['ls', '-R', str(CHAPTER / 'src')], check=True)


### 2. 一个 Tile 的队列生命周期

![TQue 管理的 Tile 生命周期](./images/tque_lifecycle.svg)

输入路径依次执行 <code>AllocTensor → DataCopy → EnQue → DeQue → Compute → FreeTensor</code>。输出路径在计算前分配 Tensor，计算后入队，再由搬出阶段出队、写回 GM 并释放。

<code>DataCopy</code> 由 MTE 异步执行。<code>EnQue/DeQue</code> 不只是容器操作，还建立 MTE 与 Vector 阶段间的依赖：消费方 DeQue 后才能安全读取上游发布的数据。<code>FreeTensor</code> 把已经消费的物理槽位交还给后续 Tile。


### 3. depth 与 num 是两个参数

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>参数</th><th style='text-align: left;'>本实验取值</th><th style='text-align: left;'>作用</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>TQue&lt;position, depth&gt;</code></td><td style='text-align: left;'><code>depth=1</code></td><td style='text-align: left;'>允许连续驻队的 Tensor 深度；本调度先 DeQue 当前 Tile，再 EnQue 下一 Tile，因此一个就够</td></tr>
    <tr><td style='text-align: left;'><code>InitBuffer(queue, num, size)</code></td><td style='text-align: left;'><code>num=1</code> 或 <code>2</code></td><td style='text-align: left;'>为队列分配的物理 Buffer 数量；<code>num=2</code> 才是双缓冲</td></tr>
    <tr><td style='text-align: left;'><code>size</code></td><td style='text-align: left;'><code>tileLength × sizeof(float)</code></td><td style='text-align: left;'>单个物理 Buffer 的有效载荷字节数</td></tr>
  </tbody>
</table>

因此不要把当前源码中的两个概念写成同一个模板参数。实验 3 的三个队列都声明为 <code>TQue&lt;..., 1&gt;</code>，只在 <code>InitBuffer</code> 中切换 <code>num</code>。


In [ ]:
source_lines = (DEMO / 'vector_add_pipeline.asc').read_text(encoding='utf-8').splitlines()
patterns = ['InitBuffer(inQueueX_', 'InitBuffer(inQueueY_', 'InitBuffer(outQueueZ_', 'TQue<AscendC::TPosition::VECIN', 'TQue<AscendC::TPosition::VECOUT']
for index, line in enumerate(source_lines, start=1):
    if any(pattern in line for pattern in patterns):
        print(f'{index:>3}: {line}')


<strong>检查点：</strong> 三个 TQue 声明的模板深度均为 <code>kQueueDepth</code>，其值固定为 1；三个 <code>InitBuffer</code> 调用的第二个参数均为 <code>kBufferNum</code>。

### 4. 计算每核队列有效载荷

本实验有两个输入队列和一个输出队列。只统计 LocalTensor 槽位的有效载荷：

<pre><code>queueBytes = 3 × bufferNum × tileLength × sizeof(float)</code></pre>

默认配置 <code>N=16384</code>、<code>blockDim=8</code>、<code>tileCount=8</code>，所以每核 <code>tileLength=256</code>、<code>tileBytes=1024</code>。


In [ ]:
def pipeline_budget(total_length, block_dim, tile_count, buffer_num):
    block_length = total_length // block_dim
    tile_length = block_length // tile_count
    tile_bytes = tile_length * 4
    return {
        'tile_count': tile_count,
        'tile_length': tile_length,
        'tile_bytes': tile_bytes,
        'buffer_num': buffer_num,
        'queue_bytes': 3 * buffer_num * tile_bytes,
    }

single = pipeline_budget(16384, 8, 8, 1)
double = pipeline_budget(16384, 8, 8, 2)
print('single:', single)
print('double:', double)
assert single['queue_bytes'] == 3072
assert double['queue_bytes'] == 6144
assert single['tile_count'] == double['tile_count'] == 8
assert double['queue_bytes'] == 2 * single['queue_bytes']


双缓冲以两倍队列有效载荷换取可重叠的两个物理槽位。本实验在 Host 侧使用 192 KiB 教学预算，并在 Kernel 启动前检查 <code>queueBytes</code>。如果生产算子的单缓冲 Tile 已接近占满 UB，改成双缓冲时必须重新计算更小的 Tile；本实验的固定用例留有足够余量，因此保持相同 Tile 做对照。

### 5. tileCount 不随 bufferNum 改变

<code>tileCount</code> 描述每个 Block 中真实的 GM Tile 数，决定完整数据覆盖；<code>bufferNum</code> 只描述可轮换使用的物理槽位。单、双缓冲默认都必须循环 8 次，不能写成 <code>tileCount / bufferNum</code>，也不能用 <code>bufferNum</code> 重新计算 GM 偏移。


## 课后代码实践

给定 <code>N=28672</code>、<code>blockDim=4</code>、<code>tileCount=7</code>、<code>float32</code>，请先手算，再补全下一单元写入的独立 Python 文件：

1. 计算 <code>tileLength</code> 和 <code>tileBytes</code>；
2. 分别计算 <code>bufferNum=1</code> 与 <code>2</code> 时三个队列的每核有效载荷；
3. 解释为什么模板 depth 仍为 1；
4. 说明把 bufferNum 改为 2 后，每核真实处理的 Tile 数和 GM 覆盖区间是否变化；
5. 写出输入 Tensor 和输出 Tensor 各自的完整队列生命周期。

代码中的函数必须校验参数，并返回 <code>3 × buffer_num × tile_length × 4</code>。初始化单元已准备当前系统用户专属的临时实践文件，下一单元通过变量绝对路径写入，不会切换 Notebook 工作目录，也不会与其他用户共享工作文件。补全 TODO 后，运行紧随 <code>%%writefile</code> 的自检单元；目标输出是单缓冲 12288 Byte、双缓冲 24576 Byte，且三个断言通过。


In [ ]:
%%writefile {PRACTICE_FILE}
def queue_payload_bytes(tile_length, buffer_num, tensor_count=3, dtype_bytes=4):
    if tile_length <= 0 or buffer_num not in (1, 2):
        raise ValueError('invalid pipeline configuration')
    # TODO：返回三个队列的每核有效载荷字节数。
    return 0


tile_length = 28672 // 4 // 7
single = queue_payload_bytes(tile_length, 1)
double = queue_payload_bytes(tile_length, 2)
print('tile_length:', tile_length)
print('single:', single)
print('double:', double)
assert single == 12288
assert double == 24576
assert double == 2 * single


In [ ]:
practice_file = PRACTICE_FILE
practice_result = subprocess.run(
    [sys.executable, str(practice_file)],
    text=True, capture_output=True, check=False,
)
print(practice_result.stdout)
if practice_result.stderr:
    print(practice_result.stderr)
if practice_result.returncode == 0:
    print('PRACTICE PASS')
else:
    print('PRACTICE TODO：starter 失败是预期现象；补全上一个单元的 TODO 后重新运行。')


### 独立完成后查看参考答案

下面的开关默认关闭。完成推导后再改为 <code>True</code>，答案通过 <code>cat</code> 在 Notebook 中展示。


In [ ]:
SHOW_ANSWER = False
if SHOW_ANSWER:
    subprocess.run(['cat', str(ANSWER)], check=True)
    print('\n--- 代码参考实现 ---')
    subprocess.run(['cat', str(ANSWER_CODE)], check=True)
else:
    print('完成练习后，将 SHOW_ANSWER 改为 True 再运行本单元。')


## 本节小结

本节明确了 queue depth、物理 Buffer 数与真实 Tile 数的边界。进入 03.03 双缓冲 VectorAdd 前，请确认你能得到默认配置的 <code>3072/6144 Byte</code>，并能解释为什么二者的 <code>tileCount</code> 都是 8。


完成后继续进入 [03.03 双缓冲 VectorAdd](03.03_double_buffer_vector_add.ipynb)。
